# পাঠ ১২ - এজেন্ট স্ক্র্যাচপ্যাড সহ চ্যাট ইতিহাস হ্রাস

এই নোটবুকটি দেখায় কীভাবে Microsoft Agent Framework ব্যবহার করে দীর্ঘ কথোপকথনে প্রসঙ্গ পরিচালনা করতে হয়। কথোপকথন বাড়ার সাথে সাথে, টোকেনের সংখ্যা বৃদ্ধি পায় — অবশেষে মডেলের প্রসঙ্গ উইন্ডো ছাড়িয়ে যায়। আমরা এটি সমাধান করি একটি **প্রসঙ্গ সারাংশ প্যাটার্ন** এবং একটি **এজেন্ট স্ক্র্যাচপ্যাড** দিয়ে যা স্থায়ী মেমোরি হিসাবে কাজ করে।

## আপনি যা শিখবেন:
১. **কেন প্রসঙ্গ ব্যবস্থাপনা গুরুত্বপূর্ণ**: টোকেন সীমা এবং প্রসঙ্গ উইন্ডো বোঝা
২. **প্রসঙ্গ-সচেতন এজেন্ট**: এমন এজেন্ট তৈরি যা তাদের নিজস্ব কথোপকথন প্রসঙ্গ পরিচালনা করে
৩. **প্রসঙ্গ সারাংশ প্যাটার্ন**: কথোপকথন ইতিহাস সংক্ষিপ্ত করার জন্য টুল ব্যবহার করা
৪. **এজেন্ট স্ক্র্যাচপ্যাড**: এমন স্থায়ী মেমোরি যা প্রসঙ্গ হ্রাসের পরে টিকে থাকে

## প্রয়োজনীয়তা:
- পরিবেশ ভেরিয়েবল কনফিগার করা সহ Azure OpenAI সেটআপ
- পূর্বের পাঠ থেকে বেসিক এজেন্ট ধারণার বোঝাপড়া


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## প্রসঙ্গ ব্যবস্থাপনা কেন গুরুত্বপূর্ণ  

প্রতিটি LLM-এর একটি সসীম **প্রসঙ্গ উইন্ডো** থাকে — একটি একক অনুরোধে এটি সর্বোচ্চ কত টোকেন প্রসেস করতে পারে। একটি বহুতরুণ কথোপকথন চলার সাথে:  

- প্রতিটি ব্যবহারকারীর বার্তা এবং সহকারী উত্তরের সাথে **টোকেন গণনা সরলরেখায় বৃদ্ধি পায়**।  
- **প্রম্পট টোকেন ব্যয়ের প্রধান কারণ** কারণ সম্পূর্ণ ইতিহাস প্রতি দফা পুনরায় পাঠানো হয়।  
- অবশেষে কথোপকথন **প্রসঙ্গ উইন্ডো অতিক্রম করে** এবং মডেল হয় ছেঁটে দেয় বা ত্রুটি দেয়।  

### প্রসঙ্গ পরিচালনার কৌশলসমূহ  

| কৌশল | এটি কিভাবে কাজ করে | বিনিময় |  
|---|---|---|  
| **ছেঁটে ফেলা** | সবথেকে পুরানো বার্তাগুলো বাদ দেয় | শুরুর প্রসঙ্গ হারায় |  
| **সারাংশ তৈরী** | পুরনো বার্তাগুলোকে একটি সারাংশে ঘনীভূত করে | কিছু বিবরণ হারায়, কিন্তু মূল বিষয় ধারণ করে |  
| **স্ক্র্যাচপ্যাড / বাহ্যিক মেমরি** | কথোপকথনের বাইরে প্রধান তথ্য সংরক্ষণ করে | সরঞ্জাম কল দরকার, কিন্তু যেকোনো হ্রাস সহ্য করে |  

এই নোটবুকটিতে আমরা **সারাংশ তৈরী** কে **স্ক্র্যাচপ্যাড সরঞ্জামের** সাথে মিলিয়ে ব্যবহার করি যেন এজেন্ট কথোপকথনের ধারাবাহিকতা বজায় রাখতে পারে যখন কথোপকথন ইতিহাস সংক্ষিপ্তকৃত হয়।  


## একটি প্রসঙ্গ-সচেতন এজেন্ট তৈরি করা


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## একটি দীর্ঘ কথোপকথন সিমুলেট করা  

আসুন একটি বহু-পর্যায়ের কথোপকথন দেখে নেওয়া যাক যাতে দেখা যায় কিভাবে প্রেক্ষাপট জমা হয়। এজেন্টকে প্রত্যেক পর্যায়ে মূল বিবরণ (পছন্দ, বাজেট, ভ্রমণের তারিখ) সংরক্ষণ করতে হবে এবং অবিচ্ছিন্নতা প্রদর্শন করতে হবে।  


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

লক্ষ্য করুন কিভাবে এজেন্ট আগের টার্ন থেকে প্রসঙ্গ বজায় রাখে — এটি জাপান, সুশি, মন্দির, পদচিত্রগ্রহণ, $3000 বাজেট, একক ভ্রমণ, এবং এপ্রিল সময়সীমা সম্পর্কে জানে। একটি সংক্ষিপ্ত আলাপচারিতায় এটি ভাল কাজ করে, কিন্তু আলাপচারিতা বৃদ্ধির সঙ্গে পুরো ইতিহাস ফেরত পাঠানো ব্যয়বহুল হয়ে ওঠে।

চলুন আরও টার্ন যুক্ত করে আলাপচারিতা চালিয়ে যাই প্রসঙ্গ সঞ্চয়ের ধারাবাহিকতা দেখতে:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## প্রসঙ্গ সারাংশ প্যাটার্ন

যেমন কথোপকথন বাড়ে, আমরা একটি **সারাংশ টুল** ব্যবহার করতে পারি জমাকৃত প্রসঙ্গকে সংক্ষিপ্ত ফর্ম্যাটে রূপান্তর করার জন্য। এজেন্ট এই টুলটি কল করে প্রধান পছন্দগুলো রেকর্ড করে যাতে পুরানো বার্তাগুলো মুছে ফেলা হলেও, গুরুত্বপূর্ণ তথ্যটি সংরক্ষিত থাকে।

এই প্যাটার্নটি আরও উন্নত ইতিহাস হ্রাসের ভিত্তি গঠন করে:
১। এজেন্ট কথোপকথন থেকে প্রধান তথ্য চিহ্নিত করে
২। এটি সারাংশ টুলটি কল করে সেগুলি সংরক্ষণ করে
৩। পুরানো বার্তাগুলো নিরাপদে অপসারণ করা যায় কারণ সারাংশ যেটা গুরুত্বপূর্ণ তা ধারণ করে

নিচে আমরা একটি `summarize_preferences` টুল সংজ্ঞায়িত করেছি যা এজেন্ট কল করে সংক্ষিপ্ত সারাংশ রেকর্ড করতে পারে যা এটি শিখেছে।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## সারসংক্ষেপ

এই পাঠে আপনাকে শেখানো হয়েছে কীভাবে Microsoft Agent Framework ব্যবহার করে দীর্ঘমেয়াদী এজেন্ট কথোপকথনে প্রসঙ্গ পরিচালনা করবেন:

### প্রধান ধারণাসমূহ
- **প্রসঙ্গ উইন্ডো সীমিত** — কথোপকথন ইতিহাসের প্রতিটি টোকেনের একটি খরচ থাকে এবং এটি সীমার দিকে গণনা করে।
- **সংক্ষিপ্তকরণ সরঞ্জাম** এজেন্টকে সংগৃহীত প্রসঙ্গকে সংক্ষিপ্ত সারাংশে পরিণত করতে দেয়, যা টোকেন ব্যবহারের পরিমাণ কমায় এবং মৌলিক তথ্য সংরক্ষণ করে।
- **এজেন্ট স্ক্র্যাচপ্যাড** প্রবাহিত কথোপকথন হ্রাসের পরও টিকে থাকা একটি স্থায়ী বহিরাগত স্মৃতি প্রদান করে।

### আপনি যা তৈরি করেছেন
- একটি **প্রসঙ্গ-সচেতন এজেন্ট** যা বহুবারের কথোপকথনে অবিচ্ছিন্নতা বজায় রাখে
- একটি **সংক্ষিপ্তকরণ সরঞ্জাম** (`summarize_preferences`) যা মূল ব্যবহারকারীর বিবরণ সংক্ষিপ্ত ফরম্যাটে রেকর্ড করে
- একটি **বহু-বার কথোপকথন** যা প্রসঙ্গ সংরক্ষণ এবং পরিবর্তন পরিচালনা প্রদর্শন করে

### বাস্তব বিশ্বের প্রয়োগসমূহ
- **গ্রাহক সেবা বট**: দীর্ঘ সমর্থন সেশনের মধ্যে পছন্দসমূহ মনে রাখে
- **ব্যক্তিগত সহকারী**: প্রসঙ্গ পুনরায় ব্যাখ্যা না করেই চলমান প্রকল্পগুলি ট্র্যাক করে
- **শিক্ষামূলক টিউটর**: বহু পারস্পরিক ক্রিয়ার মধ্যে শিক্ষার্থীর অগ্রগতি বজায় রাখে

### পরবর্তী ধাপসমূহ
- ফাইল-ভিত্তিক স্থায়িত্ব সহ একটি পূর্ণাঙ্গ স্ক্র্যাচপ্যাড টুল বাস্তবায়ন করুন
- সংক্ষিপ্তকরণের পরে স্বয়ংক্রিয় ইতিহাস সংক্ষিপ্তকরণ যোগ করুন
- অর্থবহ স্মৃতি অনুসন্ধানের জন্য ভেক্টর ডেটাবেসের সাথে একীভূত করুন
- পূর্ণ প্রসঙ্গসহ কয়েক দিনের পরে কথোপকথন পুনরায় শুরু করতে পারে এমন এজেন্ট তৈরি করুন


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
